Task 2: Database Design (SQL and MongoDB)

## Part 1: SQL Database Design (MySQL)

In [ ]:
# SQL Schema: 3 Normalized Tables

schema_sql = """
-- Table 1: assets - Reference table for financial instruments
CREATE TABLE assets (
    asset_id INT PRIMARY KEY AUTO_INCREMENT,
    asset_name VARCHAR(100) NOT NULL,
    asset_type ENUM('equity', 'bond', 'commodity', 'crypto', 'index') NOT NULL,
    ticker_symbol VARCHAR(20),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Table 2: market_data - Main time series data
CREATE TABLE market_data (
    record_id INT PRIMARY KEY AUTO_INCREMENT,
    date DATE NOT NULL,
    equities_us DECIMAL(10, 2),
    equities_tech DECIMAL(10, 2),
    equities_emerging DECIMAL(10, 2),
    bonds_longterm DECIMAL(10, 2),
    gold DECIMAL(10, 2),
    oil DECIMAL(10, 2),
    volatility_index DECIMAL(10, 2),
    crypto_bitcoin DECIMAL(12, 2),
    yield_curve_spread DECIMAL(10, 2),
    high_yield_spread DECIMAL(10, 2),
    financial_stress_index DECIMAL(10, 4),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    INDEX idx_date (date),
    INDEX idx_financial_stress (financial_stress_index)
);

-- Table 3: predictions - Store model predictions
CREATE TABLE predictions (
    prediction_id INT PRIMARY KEY AUTO_INCREMENT,
    date DATE NOT NULL,
    model_name VARCHAR(100) NOT NULL,
    predicted_value DECIMAL(10, 4),
    actual_value DECIMAL(10, 4),
    prediction_error DECIMAL(10, 4),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (date) REFERENCES market_data(date),
    INDEX idx_model (model_name),
    INDEX idx_prediction_date (date)
);
"""

print(schema_sql)

In [ ]:
# Sample data insertion
sample_data_sql = """
-- Insert sample assets
INSERT INTO assets (asset_name, asset_type, ticker_symbol) VALUES
('S&P 500', 'equity', 'SPY'),
('NASDAQ Tech', 'equity', 'QQQ'),
('Gold', 'commodity', 'GLD'),
('Bitcoin', 'crypto', 'BTC'),
('VIX', 'index', 'VIX');

-- Insert sample market data
INSERT INTO market_data (date, equities_us, gold, oil, volatility_index, crypto_bitcoin, financial_stress_index)
VALUES
('2026-02-25', 687.35, 474.61, 80.76, 19.55, 65568.49, -0.6208),
('2026-02-24', 687.35, 474.61, 80.76, 19.55, 64616.74, -0.6208),
('2026-02-23', 682.39, 481.28, 80.90, 21.01, 64616.74, -0.6208);
"""

print(sample_data_sql)

### SQL Queries

In [ ]:
# Query 1: Get latest record
query1 = """
SELECT date, equities_us, gold, volatility_index, financial_stress_index
FROM market_data
ORDER BY date DESC
LIMIT 1;
"""

print("Query 1: Latest Record")
print(query1)
print("\nExpected Result:")
print("date       | equities_us | gold   | volatility_index | financial_stress_index")
print("2026-02-25 | 687.35      | 474.61 | 19.55            | -0.6208")

In [ ]:
# Query 2: Get records by date range
query2 = """
SELECT date, equities_us, volatility_index
FROM market_data
WHERE date BETWEEN '2026-02-23' AND '2026-02-25'
ORDER BY date ASC;
"""

print("Query 2: Records by Date Range")
print(query2)
print("\nExpected Result:")
print("date       | equities_us | volatility_index")
print("2026-02-23 | 682.39      | 21.01")
print("2026-02-24 | 687.35      | 19.55")
print("2026-02-25 | 687.35      | 19.55")

In [ ]:
# Query 3: Aggregate statistics with high stress periods
query3 = """
SELECT 
    YEAR(date) as year,
    COUNT(*) as total_records,
    AVG(volatility_index) as avg_volatility,
    MAX(financial_stress_index) as max_stress,
    MIN(financial_stress_index) as min_stress
FROM market_data
WHERE financial_stress_index < -0.5
GROUP BY YEAR(date)
ORDER BY year DESC;
"""

print("Query 3: Aggregate Statistics for High Stress Periods")
print(query3)
print("\nExpected Result:")
print("year | total_records | avg_volatility | max_stress | min_stress")
print("2026 | 3             | 20.04          | -0.6208    | -0.6208")

## Part 2: MongoDB Database Design

In [ ]:
import json

# MongoDB Collection Design
collection_design = {
    "collection_name": "market_timeseries",
    "description": "Time series financial market data with embedded technical indicators",
    "sample_documents": [
        {
            "_id": "2026-02-25",
            "date": "2026-02-25T00:00:00Z",
            "equities": {
                "us": 687.35,
                "tech": 607.87,
                "emerging": 62.62
            },
            "commodities": {
                "gold": 474.61,
                "oil": 80.76
            },
            "crypto": {
                "bitcoin": 65568.49
            },
            "bonds": {
                "longterm": 89.90
            },
            "indicators": {
                "volatility_index": 19.55,
                "financial_stress_index": -0.6208,
                "yield_curve_spread": 0.61,
                "high_yield_spread": 2.95
            },
            "technical": {
                "spy_drawdown": -0.0117,
                "spy_rsi_14": 50.32,
                "gld_rsi_14": 58.72
            }
        },
        {
            "_id": "2026-02-24",
            "date": "2026-02-24T00:00:00Z",
            "equities": {
                "us": 687.35,
                "tech": 607.87,
                "emerging": 62.62
            },
            "commodities": {
                "gold": 474.61,
                "oil": 80.76
            },
            "crypto": {
                "bitcoin": 64616.74
            },
            "indicators": {
                "volatility_index": 19.55,
                "financial_stress_index": -0.6208
            }
        }
    ],
    "indexes": [
        {"field": "date", "type": "ascending"},
        {"field": "indicators.financial_stress_index", "type": "ascending"}
    ]
}

print(json.dumps(collection_design, indent=2))

### MongoDB Queries

In [ ]:
# MongoDB Query 1: Find latest record
mongo_query1 = """
db.market_timeseries.find()
  .sort({ date: -1 })
  .limit(1)
"""

print("MongoDB Query 1: Latest Record")
print(mongo_query1)
print("\nExpected Result:")
result1 = {
    "_id": "2026-02-25",
    "date": "2026-02-25T00:00:00Z",
    "equities": {"us": 687.35},
    "indicators": {"volatility_index": 19.55, "financial_stress_index": -0.6208}
}
print(json.dumps(result1, indent=2))

In [ ]:
# MongoDB Query 2: Find records by date range
mongo_query2 = """
db.market_timeseries.find({
  date: {
    $gte: ISODate("2026-02-23T00:00:00Z"),
    $lte: ISODate("2026-02-25T00:00:00Z")
  }
}).sort({ date: 1 })
"""

print("MongoDB Query 2: Records by Date Range")
print(mongo_query2)
print("\nExpected Result: 2 documents between 2026-02-23 and 2026-02-25")

In [ ]:
# MongoDB Query 3: Aggregate average volatility during high stress
mongo_query3 = """
db.market_timeseries.aggregate([
  {
    $match: {
      "indicators.financial_stress_index": { $lt: -0.5 }
    }
  },
  {
    $group: {
      _id: { $year: "$date" },
      avg_volatility: { $avg: "$indicators.volatility_index" },
      max_stress: { $max: "$indicators.financial_stress_index" },
      count: { $sum: 1 }
    }
  },
  {
    $sort: { _id: -1 }
  }
])
"""

print("MongoDB Query 3: Aggregate Statistics for High Stress Periods")
print(mongo_query3)
print("\nExpected Result:")
result3 = {
    "_id": 2026,
    "avg_volatility": 20.04,
    "max_stress": -0.6208,
    "count": 2
}
print(json.dumps(result3, indent=2))

## Summary

**SQL Database:**
- 3 normalized tables: assets, market_data, predictions
- Indexes on date and financial_stress_index
- 3 queries: latest record, date range, aggregate statistics

**MongoDB Database:**
- Embedded document structure for related data
- Nested objects for equities, commodities, indicators
- 3 queries: latest record, date range, aggregation pipeline

**Next Steps:**
- Export SQL scripts to `database/sql/schema.sql`
- Export MongoDB design to `database/mongodb/collection_design.json`
- Create ERD diagram
- Commit changes to GitHub